# Practice Set — Gradient Descent, K-Means, K-Means++ (from scratch)

**Format:** same as the Logistic Regression assignment. Complete only the cells marked `YOUR CODE HERE`.

**Rules**
- Use NumPy (and matplotlib for plots). **Do not use `sklearn`.**
- Use vectorized NumPy wherever possible. Loops are allowed only where the task says so (e.g. SGD over samples, the K-Means iteration loop).
- Always check shapes. Most exam bugs are shape bugs.

**Parts**
- Part A — Gradient Descent (Batch, Stochastic, Mini-Batch) for linear regression
- Part B — K-Means
- Part C — K-Means++

# Part A — Gradient Descent for Linear Regression

## A1. Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## A2. Load the Dataset

`X` contains two features and `y` is a continuous target. Inspect the shapes before continuing.

Let `m` = number of examples and `n` = number of features.

In [ ]:
X = np.array([
    [1, 3], [2, 1], [3, 4], [4, 2],
    [5, 6], [6, 3], [7, 7], [8, 5]
], dtype=float)

y = np.array([4.5, 5.5, 9.0, 10.0, 14.0, 14.5, 18.5, 19.5])

m, n = X.shape
print("X shape:", X.shape)
print("y shape:", y.shape)
print("m =", m, ", n =", n)

## A3. Visualize the Dataset

Make **two side-by-side subplots** (use `plt.subplots(1, 2)`):
- Left: Feature 1 vs `y`
- Right: Feature 2 vs `y`

Label the axes on each subplot.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].scatter(X[:, 0], y)
axes[0].set_xlabel("Feature 1"); axes[0].set_ylabel("y")
axes[1].scatter(X[:, 1], y)
axes[1].set_xlabel("Feature 2"); axes[1].set_ylabel("y")

plt.show()

## A4. The Model

$$\hat y = Xw + b$$

If `X.shape = (m, n)` and `w.shape = (n,)` then `X @ w` has shape `(m,)`.

**Task:** Implement `predict(X, w, b)`.

In [ ]:
def predict(X, w, b):
    return X @ w + b

In [ ]:
w = np.zeros(X.shape[1])
b = 0.0
print(predict(X, w, b))
print(predict(X, w, b).shape)
# Expected: all zeros, shape (8,)

## A5. Cost Function (Mean Squared Error)

$$J(w,b)=\frac{1}{2m}\sum_{i=1}^{m}\left(\hat y^{(i)}-y^{(i)}\right)^2$$

**Task:** Implement `compute_cost(X, y, w, b)` with no loops.

In [ ]:
def compute_cost(X, y, w, b):
    m = X.shape[0]
    errors = predict(X, w, b) - y
    return np.sum(errors ** 2) / (2 * m)

In [ ]:
print("Initial cost:", compute_cost(X, y, w, b))
# Expected: 85.015625

## A6. Gradients

$$dw=\frac{1}{m}X^{T}(\hat y-y) \qquad db=\frac{1}{m}\sum(\hat y-y)$$

**Shape check**
```
X.T.shape   = (n, m)
y_hat - y   = (m,)
dw          = (n,)     <-- NOT (n, m)
```
Do not loop over examples.

**Hint:** which operator gives a matrix-vector product, `*` or `@`?

In [ ]:
def compute_gradients(X, y, w, b):
    m = X.shape[0]
    errors = predict(X, w, b) - y
    dw = (1 / m) * (X.T @ errors)
    db = (1 / m) * np.sum(errors)
    return dw, db

In [ ]:
dw, db = compute_gradients(X, y, w, b)
print("dw:", dw)
print("dw shape:", dw.shape)
print("db:", db)
# Expected: dw = [-65.625  -53.6875], dw shape (2,), db = -11.9375

## A7. One Gradient-Descent Update

$$w := w - \alpha\, dw \qquad b := b - \alpha\, db$$

**Task:** compute the gradients, update `w` and `b`, and return them.

In [ ]:
def gradient_descent_step(X, y, w, b, learning_rate):
    dw, db = compute_gradients(X, y, w, b)
    w = w - learning_rate * dw
    b = b - learning_rate * db
    return w, b

In [ ]:
w1, b1 = gradient_descent_step(X, y, np.zeros(2), 0.0, 0.01)
print(w1, b1)
# Expected: [0.65625 0.536875] 0.119375

## A8. Complete Batch Gradient Descent

For each iteration:
1. Compute the cost and store it.
2. Take one gradient-descent step.
3. **Early stopping:** if `tol` is not `None` and the cost dropped by less than `tol` compared with the previous iteration, `break`.

Return `w, b, cost_history`.

In [ ]:
def train_bgd(X, y, learning_rate=0.01, num_iterations=1000, tol=None):
    w = np.zeros(X.shape[1])
    b = 0.0
    cost_history = []

    for i in range(num_iterations):
        cost = compute_cost(X, y, w, b)
        if tol is not None and cost_history and abs(cost_history[-1] - cost) < tol:
            cost_history.append(cost)
            break
        cost_history.append(cost)
        w, b = gradient_descent_step(X, y, w, b, learning_rate)

    return w, b, cost_history

## A9. Train the Model

Use learning rate `0.01` and `2000` iterations on the **raw** (unscaled) data.

In [ ]:
w, b, cost_history = train_bgd(X, y, 0.01, 2000)
print("Final weights:", w)
print("Final bias:", b)
print("Final cost:", cost_history[-1])
# Expected: w ~ [2.00 0.50], b ~ 0.97, cost ~ 6e-05

## A10. Plot the Cost

Plot iteration number against cost.

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(cost_history)
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.title("Batch Gradient Descent")
plt.show()

## Test your understanding — Learning Rate Experiment

Train with `learning_rate` in `[0.001, 0.01, 0.05]` for `2000` iterations on the raw data. Print the final cost for each.

**Question 1.** Which learning rate is most appropriate and why? What exactly went wrong with the others?

**Question 2.** What would you see on the cost-vs-iteration plot for a learning rate that is too large?

In [ ]:
learning_rates = [0.001, 0.01, 0.05]

import warnings
warnings.filterwarnings("ignore")
for lr in learning_rates:
    _, _, h = train_bgd(X, y, lr, 2000)
    print(f"lr = {lr:<6} final cost = {h[-1]}")

> ### ✅ Solution / Answer

**Answer 1.** `0.01` is the best of the three (final cost ~6e-05). `0.001` is *too small*: the cost is still decreasing after 2000 iterations (final cost ~0.027, weights not converged yet). `0.05` is *too large*: each step overshoots the minimum, the error grows instead of shrinking, and the cost blows up (overflow / `inf` / `nan`).

**Answer 2.** The cost curve **increases** (or oscillates with growing amplitude) instead of decreasing, and eventually explodes to `inf`/`nan`.

## A11. Feature Standardization

The two features have different ranges, which forces a small learning rate. Standardize each column:

$$x_{scaled}=\frac{x-\mu}{\sigma}$$

**Task:** implement `standardize(X)` returning `X_scaled, mu, sigma` (column-wise, shape `(n,)` each).

**Important:** at prediction time on new data you must reuse the **training** `mu` and `sigma`.

In [ ]:
def standardize(X):
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    X_scaled = (X - mu) / sigma
    return X_scaled, mu, sigma

In [ ]:
X_scaled, mu, sigma = standardize(X)
print("mu:", mu)
print("sigma:", sigma)
print("column means after scaling:", X_scaled.mean(axis=0))
print("column stds after scaling: ", X_scaled.std(axis=0))
# Expected: mu = [4.5 3.875], sigma = [2.2913 1.8998], means ~0, stds ~1

## A12. Train on Standardized Data

Train `train_bgd` on `X_scaled` with `learning_rate = 0.1` and `200` iterations.

**Question 3.** Learning rate `0.1` diverged on the raw data (`0.05` already did). Why does it converge now?

**Question 4.** The learned weights are now about `[4.58, 0.95]` instead of `[2.0, 0.5]`. Is the model wrong? Explain.

**Question 5.** The learned bias is about `11.94`. What simple statistic of `y` is this equal to, and why?

In [ ]:
w_s, b_s, h_s = train_bgd(X_scaled, y, 0.1, 200)
print("weights:", w_s)
print("bias:", b_s)
print("final cost:", h_s[-1])
print("mean of y:", y.mean())

> ### ✅ Solution / Answer

**Answer 3.** On raw data the features have large, different scales, so the cost surface is a long narrow valley: the steepest direction is very steep, which forces a tiny learning rate to avoid overshooting. After standardization all features have similar scale (mean 0, std 1), the surface is closer to round, and a much larger stable learning rate works and converges faster.

**Answer 4.** No, the model is equivalent. It is fit in *standardized* feature units. Converting back: `w_original = w_scaled / sigma` and `b_original = b_scaled - sum(w_scaled * mu / sigma)`, which gives `[2.0, 0.5]` and `1.0`.

**Answer 5.** `b = mean(y) = 11.9375`. With standardized features every feature has mean 0, so `X_scaled @ w` averages to 0 over the data, and the optimal bias is just the average target.

## A13. Predict a New Point with the Standardized Model

Predict `y` for the new point `[10, 4]`. Use the model trained in A12. The true value from the underlying rule is `23`.

In [ ]:
x_new = np.array([10, 4], dtype=float)
x_new_scaled = (x_new - mu) / sigma          # training mu and sigma, NOT recomputed
print(x_new_scaled @ w_s + b_s)             # ~23

## A14. Stochastic Gradient Descent (SGD)

Update the parameters after **every single example**.

For each epoch:
1. Shuffle the indices (`rng.permutation(m)`).
2. For each index `i`: compute `error = x_i @ w + b - y_i`, then `w -= lr * error * x_i` and `b -= lr * error`.
3. After the epoch, store the cost on the **full** dataset.

Return `w, b, cost_history` (one cost per epoch). Use `rng = np.random.default_rng(seed)`.

In [ ]:
def train_sgd(X, y, learning_rate=0.05, num_epochs=100, seed=0):
    rng = np.random.default_rng(seed)
    m = X.shape[0]
    w = np.zeros(X.shape[1])
    b = 0.0
    cost_history = []

    for epoch in range(num_epochs):
        for i in rng.permutation(m):
            error = X[i] @ w + b - y[i]
            w = w - learning_rate * error * X[i]
            b = b - learning_rate * error
        cost_history.append(compute_cost(X, y, w, b))

    return w, b, cost_history

In [ ]:
w_sgd, b_sgd, h_sgd = train_sgd(X_scaled, y, 0.05, 100)
print(w_sgd, b_sgd, h_sgd[-1])
# Expected: w ~ [4.58 0.95], b ~ 11.94, cost very close to 0

## A15. Mini-Batch Gradient Descent

Same idea, but each update uses a batch of `batch_size` examples.

For each epoch: shuffle, split into consecutive batches (the last batch may be smaller), compute the **batch-average** gradient, update. Store the full-dataset cost after each epoch.

In [ ]:
def train_minibatch(X, y, learning_rate=0.1, num_epochs=200, batch_size=4, seed=0):
    rng = np.random.default_rng(seed)
    m = X.shape[0]
    w = np.zeros(X.shape[1])
    b = 0.0
    cost_history = []

    for epoch in range(num_epochs):
        perm = rng.permutation(m)

        for start in range(0, m, batch_size):
            idx = perm[start:start + batch_size]
            Xb, yb = X[idx], y[idx]
            w, b = gradient_descent_step(Xb, yb, w, b, learning_rate)   # averages over len(idx)

        cost_history.append(compute_cost(X, y, w, b))

    return w, b, cost_history

In [ ]:
w_mb, b_mb, h_mb = train_minibatch(X_scaled, y, 0.1, 200, batch_size=4)
print(w_mb, b_mb, h_mb[-1])

## A16. Compare the Three Methods

Plot the cost history of Batch GD (A12), SGD (A14) and Mini-Batch (A15) on one figure, using a **log** y-axis (`plt.yscale("log")`). Add a legend.

**Question 6.** Compare BGD, SGD and Mini-Batch on: (a) cost per update, (b) smoothness of the loss curve, (c) updates per epoch, (d) memory / speed on huge datasets.

**Question 7.** With a fixed learning rate, SGD never settles exactly at the minimum on noisy data. Why, and what is the standard fix?

**Question 8.** In this notebook the SGD cost is recorded once per epoch on the full data, but the BGD cost is recorded once per iteration. Why is it wrong to compare the x-axes directly?

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(h_s, label="Batch GD")
plt.plot(h_sgd, label="SGD")
plt.plot(h_mb, label="Mini-batch")
plt.yscale("log")
plt.xlabel("Epoch / iteration")
plt.ylabel("Cost (log scale)")
plt.legend()
plt.show()

> ### ✅ Solution / Answer

**Answer 6.**
| | Batch | SGD | Mini-batch |
|---|---|---|---|
| Data per update | all `m` | 1 sample | `b` samples |
| Gradient | exact | very noisy | moderately noisy |
| Loss curve | smooth, monotone | jumpy | in between |
| Updates per epoch | 1 | `m` | `ceil(m/b)` |
| Large data | slow / heavy on memory per step | cheap steps, can stream data | best trade-off, vectorizes well on GPU/CPU |

**Answer 7.** Each update uses one sample's gradient, which is a noisy estimate of the true gradient. Even at the optimum, individual sample gradients are non-zero (when the data is noisy), so the parameters keep bouncing around the minimum. Fix: **decay the learning rate** over time (or use mini-batches / momentum).

**Answer 8.** One BGD iteration = one update = one full pass over the data. One SGD epoch = `m` updates. So one SGD epoch does much more parameter movement than one BGD iteration; compare per epoch (passes over data) or per wall-clock time, not per index.

## A17. Conceptual Questions (write short answers)

**Q9.** Why is the cost divided by `2m` instead of `m`? Does it change the location of the minimum?

**Q10.** Gradient descent uses `w := w - alpha * dw` with a **minus** sign. What would happen with a plus sign?

**Q11.** Suppose the dataset had `1,000,000` rows. Which variant would you use and why?

**Q12.** What are two stopping criteria for gradient descent other than a fixed number of iterations?

**Q13.** A student writes `dw = (1/m) * X.T * (y_hat - y)` in the code (elementwise `*`). What shape does `dw` have and why is it wrong?

> ### ✅ Solution / Answer

**Q9.** The `1/2` cancels the `2` that comes from differentiating the square, giving `dw = (1/m) X^T (y_hat - y)`. A constant factor never moves the minimum, so the optimal `w, b` is identical.

**Q10.** With `+` you move *up* the gradient (gradient **ascent**): the cost increases and diverges.

**Q11.** Mini-batch (or SGD): full-batch needs a pass over all million rows for a single update; mini-batch makes many cheap, vectorized updates per epoch and converges much sooner in wall-clock time.

**Q12.** (i) Cost change between iterations below a tolerance (`|J_prev - J| < tol`); (ii) gradient norm `||dw||` below a threshold; (iii) validation cost stops improving (early stopping).

**Q13.** `X.T` is `(n, m)` and `(y_hat - y)` is `(m,)`, so `*` broadcasts to `(n, m)` instead of reducing over samples. The correct `X.T @ (y_hat - y)` gives shape `(n,)`. (This is exactly the shape bug in the logistic regression assignment.)

# Part B — K-Means

## B1. Load the Dataset

Nine 2-D points. We will use `K = 3`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X = np.array([
    [1.0, 2.0], [1.5, 1.8], [5.0, 8.0],
    [8.0, 8.0], [1.0, 0.6], [9.0, 11.0],
    [8.0, 2.0], [10.0, 2.0], [9.0, 3.0]
])

K = 3
initial_centroids = X[[0, 2, 6]].copy()      # points 0, 2 and 6

print("X shape:", X.shape)
print("initial_centroids:\n", initial_centroids)

## B2. Visualize the Dataset

Scatter plot of `X`. Mark the three initial centroids with a red `X` marker (`marker='X'`, `s=200`).

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], s=60)
plt.scatter(initial_centroids[:, 0], initial_centroids[:, 1], marker='X', color='red', s=200)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("K-Means Dataset")
plt.show()

## B3. Euclidean Distance

$$d(p,c)=\sqrt{\sum_{j}(p_j-c_j)^2}$$

**Task:** implement `euclidean_distance(point, centroid)` for a **single** point and centroid.

In [ ]:
def euclidean_distance(point, centroid):
    return np.sqrt(np.sum((point - centroid) ** 2))

In [ ]:
print(euclidean_distance(np.array([4, 6]), np.array([1, 2])))
# Expected: 5.0

## B4. Distance Matrix (no loops)

**Task:** implement `compute_distances(X, centroids)` returning an array of shape `(m, K)` where entry `[i, k]` is the distance from point `i` to centroid `k`. **No loops.**

**Hint:** use broadcasting.
```
X[:, None, :]          shape (m, 1, n)
centroids[None, :, :]  shape (1, K, n)
difference             shape (m, K, n)  -> reduce over the last axis
```

In [ ]:
def compute_distances(X, centroids):
    diff = X[:, None, :] - centroids[None, :, :]      # (m, K, n)
    return np.sqrt(np.sum(diff ** 2, axis=2))         # (m, K)

In [ ]:
D = compute_distances(X, initial_centroids)
print(D.round(3))
print("shape:", D.shape)
# Expected shape: (9, 3); first row = [0. 7.211 7.]

## B5. Assign Clusters

**Task:** implement `assign_clusters(X, centroids)` returning an integer array `idx` of shape `(m,)` with the index of the closest centroid for each point. **No loops** (use `np.argmin` with an `axis`).

In [ ]:
def assign_clusters(X, centroids):
    return np.argmin(compute_distances(X, centroids), axis=1)

In [ ]:
idx = assign_clusters(X, initial_centroids)
print(idx)
# Expected: [0 0 1 1 0 1 2 2 2]

## B6. Update Centroids

For each cluster, set the new centroid to the **mean of its points**.

**Edge case:** if a cluster receives no points, keep its old centroid (otherwise `np.mean` of an empty array returns `nan`).

Return an array of shape `(K, n)`.

In [ ]:
def compute_centroids(X, idx, centroids):
    K = centroids.shape[0]
    new_centroids = centroids.copy()

    for k in range(K):
        points = X[idx == k]
        if len(points) > 0:
            new_centroids[k] = points.mean(axis=0)

    return new_centroids

In [ ]:
new_centroids = compute_centroids(X, idx, initial_centroids)
print(new_centroids.round(4))
# Expected: [[1.1667 1.4667] [7.3333 9.] [9. 2.3333]]

## B7. Cost (Distortion)

$$J=\frac{1}{m}\sum_{i=1}^{m}\left\|x^{(i)}-\mu_{c^{(i)}}\right\|^2$$

The **squared** distance from each point to the centroid of its own cluster, averaged over all points.

**Task:** implement `compute_cost(X, idx, centroids)` **without loops**. (Hint: `centroids[idx]` has shape `(m, n)`.)

In [ ]:
def compute_cost(X, idx, centroids):
    return np.mean(np.sum((X - centroids[idx]) ** 2, axis=1))

In [ ]:
print(compute_cost(X, idx, new_centroids))
# Expected: 2.0718518...

## B8. One Full Iteration

Combine assignment, centroid update and cost. Return `idx, new_centroids, cost`.

In [ ]:
def run_iteration(X, centroids):
    idx = assign_clusters(X, centroids)
    new_centroids = compute_centroids(X, idx, centroids)
    cost = compute_cost(X, idx, new_centroids)
    return idx, new_centroids, cost

In [ ]:
idx, new_centroids, cost = run_iteration(X, initial_centroids)
print(idx, cost)

## B9. Complete K-Means

Repeat until convergence or `max_iters`:
1. Assign clusters.
2. Update centroids.
3. Store the cost.
4. If the centroids did not change (`np.allclose`), `break`.

Return `centroids, idx, cost_history`.

In [ ]:
def run_kmeans(X, initial_centroids, max_iters=100):
    centroids = initial_centroids.copy()
    cost_history = []

    for i in range(max_iters):
        idx = assign_clusters(X, centroids)
        new_centroids = compute_centroids(X, idx, centroids)
        cost_history.append(compute_cost(X, idx, new_centroids))
        if np.allclose(new_centroids, centroids):
            break
        centroids = new_centroids

    idx = assign_clusters(X, centroids)
    return centroids, idx, cost_history

In [ ]:
final_centroids, idx, cost_history = run_kmeans(X, initial_centroids)
print("Final centroids:\n", final_centroids.round(4))
print("Assignments:", idx)
print("Cost history:", np.round(cost_history, 4))

## B10. Visualize the Result

Plot the points coloured by cluster (`c=idx`) and the final centroids as red `X` markers.

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=idx, s=60, cmap='tab10')
plt.scatter(final_centroids[:, 0], final_centroids[:, 1], marker='X', color='red', s=200)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("K-Means Result")
plt.show()

## B11. Plot the Cost per Iteration

Plot `cost_history`.

**Question 1.** Is the cost ever allowed to increase between iterations? Explain using the two steps of the algorithm.

**Question 2.** Why does the cost history above contain the same value twice at the end?

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(cost_history) + 1), cost_history, marker='o')
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.title("K-Means Cost")
plt.show()

> ### ✅ Solution / Answer

**Answer 1.** No. K-Means is coordinate descent on `J`. The **assignment step** picks, for fixed centroids, the closest centroid for each point, which can only lower (or keep) `J`. The **update step** picks, for fixed assignments, the mean, which is the minimizer of the sum of squared distances, so it also can only lower (or keep) `J`. Hence `J` is non-increasing.

**Answer 2.** In the last iteration the assignments (and hence centroids) did not change, so the cost is identical to the previous one; that is precisely the convergence condition that triggers `break`.

## B12. Elbow Method with Multiple Restarts

K-Means depends on the initialization. To get a reliable cost for each `K`, run it `n_init` times from **random** initial centroids (choose `K` distinct data points using `rng.choice(m, K, replace=False)`) and keep the **lowest** final cost.

**Task:** implement `best_cost(X, K, n_init=10, seed=0)`, compute it for `K = 1..6`, and plot cost vs `K`.

**Question 3.** Which `K` would you choose and why?

**Question 4.** Why can we not simply pick the `K` with the lowest cost?

In [ ]:
def best_cost(X, K, n_init=10, seed=0):
    rng = np.random.default_rng(seed)
    best = np.inf

    for _ in range(n_init):
        init = X[rng.choice(len(X), K, replace=False)]
        cents, idx_, _ = run_kmeans(X, init)
        cost_ = compute_cost(X, idx_, cents)
        best = min(best, cost_)

    return best

In [ ]:
ks = range(1, 7)
costs = [best_cost(X, k) for k in ks]

plt.figure(figsize=(6, 4))
plt.plot(list(ks), costs, marker='o')
plt.xlabel("K")
plt.ylabel("Cost")
plt.title("Elbow Method")
plt.show()
print(np.round(costs, 4))

> ### ✅ Solution / Answer

**Answer 3.** `K = 3`. The cost drops steeply from K=1 (24.75) to K=2 (9.94) to K=3 (2.07) and then flattens (0.94, 0.44, 0.26): the "elbow" is at 3. This matches the three visible groups in the scatter plot.

**Answer 4.** The cost is monotonically non-increasing in `K` and reaches `0` at `K = m` (every point is its own centroid). Minimizing cost alone always favours over-fitting with too many clusters, so we look for the point of diminishing returns instead.

## B13. Hand Calculation (do this on paper, then verify with code)

Points and initial centroids:
```
X  = [[1,1], [2,1], [4,3], [5,4], [6,5], [2,3]]
c0 = [1,1]    c1 = [6,5]
```
(a) Assign each point to a centroid. (b) Compute the new centroids. (c) Compute the cost **after** the update (mean squared distance). (d) Compute the cost **before** the update, using the old centroids. (e) Will the next iteration change anything?

Now verify with your functions.

In [ ]:
X_h = np.array([[1,1],[2,1],[4,3],[5,4],[6,5],[2,3]], dtype=float)
c_h = np.array([[1,1],[6,5]], dtype=float)

idx_h = assign_clusters(X_h, c_h)
new_h = compute_centroids(X_h, idx_h, c_h)
print("assignments:", idx_h)
print("new centroids:\n", new_h.round(4))
print("cost after update :", compute_cost(X_h, idx_h, new_h))
print("cost before update:", compute_cost(X_h, idx_h, c_h))
print("second assignment:", assign_clusters(X_h, new_h))

> ### ✅ Solution / Answer

**Answer.**
(a) `[0, 0, 1, 1, 1, 0]`  (points 0, 1, 5 go to c0; points 2, 3, 4 go to c1)
(b) `c0 = mean([1,1],[2,1],[2,3]) = [1.667, 1.667]`, `c1 = mean([4,3],[5,4],[6,5]) = [5, 4]`
(c) cost after update = **1.2222**
(d) cost before update = **2.6667** (so the update step lowered the cost)
(e) No. The second assignment is identical, so the algorithm has converged after one iteration.

## B14. Conceptual Questions

**Q5.** Does K-Means always find the global minimum of the cost? What is the practical remedy?

**Q6.** What happens if you run the assignment step and one centroid ends up with zero points? How does your code handle it?

**Q7.** Why does feature scaling matter for K-Means?

**Q8.** What is the time complexity of one K-Means iteration in terms of `m`, `K`, `n`?

**Q9.** Why do we use the *mean* in the update step and not the median or the mode?

**Q10.** Give one situation where K-Means performs poorly regardless of initialization.

> ### ✅ Solution / Answer

**Q5.** No. The cost is non-convex, so K-Means only converges to a *local* minimum that depends on the initial centroids. Remedies: run several random restarts and keep the lowest cost (`n_init`), and/or use K-Means++ initialization.

**Q6.** The empty cluster's mean is undefined (`nan`). Our `compute_centroids` keeps the old centroid for that cluster. Other options are re-seeding it at the point farthest from its centroid, or randomly.

**Q7.** Distances are dominated by the feature with the largest range (e.g. salary in the tens of thousands vs age in the tens), so the clustering effectively ignores the small-scale features. Standardize first.

**Q8.** `O(m * K * n)` per iteration (distance from every point to every centroid, each of dimension `n`).

**Q9.** The mean is the point that minimizes the sum of *squared* Euclidean distances to the cluster, which is exactly the K-Means cost. (The median minimizes the sum of absolute distances, which is K-Medians/Manhattan.)

**Q10.** Non-spherical / very different-size clusters (concentric rings, long crescents), clusters with very different densities, or data with outliers that drag centroids. K-Means assumes roughly spherical, similarly sized blobs.

# Part C — K-Means++

K-Means++ chooses the initial centroids more carefully:

1. Choose the **first** centroid uniformly at random from the data points.
2. For every point compute $D(x)^2$ = squared distance to the **nearest already-chosen** centroid.
3. Choose the next centroid with probability $P(x)=\dfrac{D(x)^2}{\sum_j D(x_j)^2}$.
4. Repeat 2 and 3 until `K` centroids are chosen. Then run ordinary K-Means.

## C1. Hand Calculation of the Selection Probabilities

Suppose the first centroid chosen is `X[0] = [1.0, 2.0]` (the dataset from Part B).

**Task:** compute `d2`, the squared distance of every point to this centroid, and the selection probability `probs` for the second centroid. Print both.

**Question 1.** What is the probability that `X[0]` is picked again? Why?

**Question 2.** Which point is the most likely second centroid, and why is this good?

In [ ]:
first = X[0:1]          # keep the 2-D shape (1, n)

d2 = np.sum((X - first) ** 2, axis=1)
probs = d2 / d2.sum()
print("d2   :", d2.round(2))
print("probs:", probs.round(4))
print("sum of probs:", probs.sum())

> ### ✅ Solution / Answer

Expected: `d2 = [0, 0.29, 52, 85, 1.96, 145, 49, 81, 65]`, `probs ≈ [0, 0.0006, 0.1085, 0.1774, 0.0041, 0.3026, 0.1022, 0.1690, 0.1356]`.

**Answer 1.** Exactly `0`, because its distance to itself is `0`, so `D² = 0`. (The same holds for any point coinciding with an already-chosen centroid; K-Means++ never picks the same point twice when points are distinct.)

**Answer 2.** Point 5 `[9, 11]` (probability ≈ 0.30), the farthest point. The rule favours points far from the existing centroids, which spreads the initial centroids across the data instead of putting two of them inside the same cluster.

## C2. Implement K-Means++ Initialization

**Task:** implement `kmeans_pp_init(X, K, rng)` that returns the `(K, n)` array of initial centroids. Use `rng.integers(m)` for the first centroid and `rng.choice(m, p=probs)` for the others. Compute distances to the **nearest** chosen centroid at every step (reuse `compute_distances`).

In [ ]:
def kmeans_pp_init(X, K, rng):
    m = X.shape[0]
    centroids = [X[rng.integers(m)]]

    for _ in range(1, K):
        d2 = np.min(compute_distances(X, np.array(centroids)) ** 2, axis=1)   # nearest chosen centroid
        probs = d2 / d2.sum()
        centroids.append(X[rng.choice(m, p=probs)])

    return np.array(centroids)

In [ ]:
rng = np.random.default_rng(0)
init_pp = kmeans_pp_init(X, 3, rng)
print(init_pp)
print("shape:", init_pp.shape)
# Expected shape: (3, 2), three distinct rows taken from X

## C3. Random Initialization vs K-Means++

Run K-Means `200` times with each initialization method (`K = 3`) and record the **final cost** of every run.
- Random: `X[rng.choice(m, 3, replace=False)]`
- K-Means++: `kmeans_pp_init(X, 3, rng)`

Print for each method: mean cost, min cost, max cost, and the **fraction of runs** that ended in a worse local minimum than the best cost found.

**Question 3.** Which method is better and why? Is K-Means++ guaranteed to reach the global minimum?

**Question 4.** What is the extra cost of K-Means++ initialization compared with random initialization?

In [ ]:
def trial_costs(init_fn, n_trials=200, seed=1):
    rng = np.random.default_rng(seed)
    costs = []

    for _ in range(n_trials):
        init = init_fn(X, 3, rng)
        cents, idx_, _ = run_kmeans(X, init)
        costs.append(compute_cost(X, idx_, cents))

    return np.array(costs)

In [ ]:
random_init = lambda X, K, rng: X[rng.choice(len(X), K, replace=False)]

rand_costs = trial_costs(random_init)
pp_costs = trial_costs(kmeans_pp_init)
best_found = min(rand_costs.min(), pp_costs.min())

for name, c in [("Random", rand_costs), ("K-Means++", pp_costs)]:
    bad = np.mean(c > best_found + 1e-9)
    print(f"{name:10s} mean={c.mean():.3f} min={c.min():.3f} max={c.max():.3f}  stuck-in-bad-minimum={bad:.1%}")

> ### ✅ Solution / Answer

Typical result (200 trials): Random mean ≈ 3.68 with ≈ 19% of runs stuck in a bad local minimum (cost ≈ 9–11); K-Means++ mean ≈ 2.20 with only ≈ 1–2% bad runs. Both reach the best cost `2.0719` when they succeed.

**Answer 3.** K-Means++ is better: because it spreads the initial centroids apart, it rarely starts with two centroids in the same natural cluster, so it converges to a good minimum more often and in fewer iterations. It is **not** guaranteed to find the global minimum, only that the *expected* cost is within `O(log K)` of optimal.

**Answer 4.** A little: `K - 1` extra passes over the data to compute `D²`, i.e. `O(m * K * n)` in total, roughly the cost of one K-Means iteration. It is usually repaid by fewer iterations to converge and better clusters.

## C4. Conceptual Questions

**Q5.** Why is the *squared* distance used as the selection weight instead of the plain distance?

**Q6.** What happens in `kmeans_pp_init` if all data points are identical (or if `K` is larger than the number of distinct points)? How could you guard against it?

**Q7.** In C2, why do we use the distance to the **nearest** chosen centroid and not the sum of distances to all chosen centroids?

**Q8.** After K-Means++ initialization, is the rest of the algorithm different from ordinary K-Means?

**Q9.** Suppose `K = 1`. What does K-Means++ return, and what is the final cluster centroid?

> ### ✅ Solution / Answer

**Q5.** Squaring increases the preference for far-away points (a point twice as far is four times as likely) and matches the K-Means objective, which is a sum of *squared* distances. This is what gives the `O(log K)` approximation guarantee.

**Q6.** All `D² = 0`, so `probs = d2 / d2.sum()` divides by zero and returns `nan`. Guard: if `d2.sum() == 0`, fall back to a uniform random choice (or stop / reduce `K`).

**Q7.** A point is "covered" if *some* centroid is already close to it. The sum would make points look far just because they are far from an unrelated centroid, even when they already sit right on top of one. The minimum measures how poorly each point is currently represented.

**Q8.** No. Only the initialization differs; assignment, update and convergence check are identical.

**Q9.** It returns a single uniformly random data point (no second-stage sampling); K-Means then moves it to the mean of the whole dataset in one update, so the final centroid is `X.mean(axis=0)`, independent of initialization.

---
### Self-check before the exam
- Can you write `sigmoid`-style one-liners from memory: `X @ w + b`, `X.T @ err / m`, `np.argmin(D, axis=1)`, `np.mean(X[idx == k], axis=0)`?
- Can you state the **shape** of every array you create?
- Can you explain, in one sentence each: why LR too large diverges, why scale features, why K-Means cost never increases, why K-Means++ helps?